# Case Study: GenAI-Powered Fraud Detection System
### Objective
The goal of this project is to build an automated, unsupervised fraud detection system that monitors transactions, identifies anomalies, and uses a GenAI (LLM) to provide reasoning for each flagged activity.

## Task 1: Data Collection & Preprocessing
We begin by loading the bank transaction data and preparing it for machine learning analysis. This involves formatting dates, encoding categorical labels, and scaling numerical values.

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import LabelEncoder, StandardScaler
import plotly.express as px
import ipywidgets as widgets
from datetime import datetime

# Load Data
df = pd.read_csv('bank_transactions_with_anomalies.csv')
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['Hour'] = df['TransactionDate'].dt.hour
df['DayOfWeek'] = df['TransactionDate'].dt.dayofweek

# Show sample data
df.head()

,TransactionID,AccountID,TransactionAmount,TransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,PreviousTransactionDate,Hour,DayOfWeek,Anomaly_Score,Is_Anomaly
0,TX000001,AC00128,14.09,2023-04-11 16:29:00,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,04-11-2024 08:08,16,1,1,0
1,TX000002,AC00455,376.24,2023-06-27 16:44:00,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,04-11-2024 08:09,16,1,1,0
2,TX000003,AC00019,126.29,2023-07-10 18:16:00,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,1,1122.35,04-11-2024 08:07,18,0,1,0
3,TX000004,AC00070,184.50,2023-05-05 16:32:00,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,1,8569.06,04-11-2024 08:09,16,4,1,0
4,TX000005,AC00411,13.45,2023-10-16 17:51:00,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,04-11-2024 08:06,17,0,-1,1


## Task 2: Anomaly Detection using Unsupervised ML
Since we do not have labels for fraud, we use **Isolation Forest**, an unsupervised model that detects anomalies by isolating them in the data space. Anomalies are points that are 'easier' to isolate than normal points.

In [3]:
# Visualization of Anomalies
fig = px.scatter(df, x='TransactionAmount', y='CustomerAge', color='Is_Anomaly', 
                 title='Transaction Amount vs Customer Age (Anomalies in Yellow)',
                 color_continuous_scale=['blue', 'red'],
                 hover_data=['TransactionID', 'Location', 'TransactionType'])
fig.show()

## Task 3: GenAI-Powered Contextual Dashboard
Here we implement a simple dashboard using `ipywidgets` (which works inside Jupyter). When a flagged Transaction ID is selected, the dashboard provides the reasoning (simulating a GenAI API call).

In [4]:
# Mock LLM Reasoning (Simulated API call)
def get_llm_explanation(transaction_id):
    row = df[df['TransactionID'] == transaction_id].iloc[0]
    # Real GenAI prompt construction:
    # prompt = f"Analyze this transaction: {row.to_dict()} and explain why it is suspicious."
    # For this project, we've pre-computed the explanations using rules and heuristics.
    return row['Reasoning']

# Dashboard UI
anomaly_ids = df[df['Is_Anomaly'] == 1]['TransactionID'].tolist()
dropdown = widgets.Dropdown(options=anomaly_ids, description='Flagged ID:')
output_text = widgets.Output()

def on_change(change):
    with output_text:
        output_text.clear_output()
        explanation = get_llm_explanation(change['new'])
        print(f"GenAI Explanation: {explanation}")
        display(df[df['TransactionID'] == change['new']])

dropdown.observe(on_change, names='value')
display(dropdown, output_text)

Dropdown(description='Flagged ID:', options=('TX000005', 'TX000020', 'TX000027', 'TX000043', 'TX000086', 'TX00…

Output()

## Task 4 & 5: Automation & Workflow
This section outlines the automated data pipeline: 
1. **Trigger**: New transactions added to the database.
2. **Process**: Preprocessing and `IsolationForest` scoring script.
3. **GenAI Filtering**: High-score anomalies sent to LLM for summary.
4. **Reporting**: Send anomalies with reasoning to a daily Excel/Dashboard report for investigators.